[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/notebooks/EN/chapter2_seminar_notebook.ipynb)

# Modelling Financial Markets — Seminar 2: Market Efficiency

*Seminar notebook. Daniel Traian Pele, Antoaneta Amza, Bucharest University of Economic Studies.*

- **[Solved]** problems contain the full solution; **[Proposed]** problems are solved by you.
- Part A: pen and paper (A1–A6); Part B: computation and inference (B1–B7); Part C: open question.

> The solutions are discussed in the seminar.

In [ ]:
import os
import re
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from scipy.special import gammaln
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
GROUP_COL = {'Developed': MainBlue, 'Emerging/frontier': IDAred, 'Crypto': Amber, 'FX': Forest}

SEED = 42
TABLE_DIR = '.'


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)



import tempfile
TABLE_DIR = tempfile.gettempdir()   # tables are written to a temporary folder


def save_fig(name):
    """Show the figure."""
    plt.show()
    plt.close()

## Data

- Daily closes up to 18 September 2026.
- Equity indices: weekdays only, days with an unchanged close (filled holidays) removed.
- Crypto trades 7 days a week; EUR/RON is the BNR official reference rate.
- Each series is analysed on its own calendar.

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# copia locala a folderului data/market, daca notebook-ul ruleaza din repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'

# nume -> (simbol, eticheta, grup, data de start, zile pe an)
MARKETS = {
    'sp500':  ('GSPC.INDX',     'S&P 500 (US)',          'Developed', '2000-01-01', 252),
    'stoxx':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Developed', '2000-01-01', 252),
    'nikkei': ('N225.INDX',     'Nikkei 225 (Japan)',    'Developed', '2000-01-01', 252),
    'hsi':    ('HSI.INDX',      'Hang Seng (HK)',        'Developed', '2000-01-01', 252),
    'bet':    ('BET',           'BET (Romania)',         'Emerging/frontier', '2000-01-01', 252),
    'wig20':  ('WIG20.INDX',    'WIG20 (Poland)',        'Emerging/frontier', '2000-01-01', 252),
    'bux':    ('BUX.INDX',      'BUX (Hungary)',         'Emerging/frontier', '2000-01-01', 252),
    'px':     ('PX.INDX',       'PX (Czechia)',          'Emerging/frontier', '2000-01-01', 252),
    'xu100':  ('XU100.INDX',    'BIST 100 (Turkey)',     'Emerging/frontier', '2000-01-01', 252),
    'bvsp':   ('BVSP.INDX',     'Bovespa (Brazil)',      'Emerging/frontier', '2000-01-01', 252),
    'mxx':    ('MXX.INDX',      'IPC (Mexico)',          'Emerging/frontier', '2000-01-01', 252),
    'nsei':   ('NSEI.INDX',     'Nifty 50 (India)',      'Emerging/frontier', '2000-01-01', 252),
    'ssec':   ('SSEC.INDX',     'Shanghai (China)',      'Emerging/frontier', '2000-01-01', 252),
    'btc':    ('BTC-USD.CC',    'Bitcoin',               'Crypto', '2014-09-17', 365),
    'eth':    ('ETH-USD.CC',    'Ethereum',              'Crypto', '2016-01-01', 365),
    'eurron': ('REF:EUR',       'EUR/RON (reference rate)', 'FX', '2005-07-01', 252),
}

LABELS = {k: v[1] for k, v in MARKETS.items()}
GROUPS = {k: v[2] for k, v in MARKETS.items()}
PERIODS = {k: v[4] for k, v in MARKETS.items()}

_CACHE = {}


def read_market(symbol):
    """Citeste data/market/<SIMBOL>.csv local sau din repo-ul GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Cursul oficial de referinta RON (arhive XML anuale BNR)."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END):
    """Pretul de inchidere zilnic, curatat dupa conventiile capitolului."""
    symbol, _, group, start0, _ = MARKETS[name]
    start = start or start0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    s = read_market(symbol)['close'].loc[start:end]
    s = s[s > 0].dropna()
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # fara cotatii de weekend
        s = s[s.diff() != 0]                  # fara sarbatori completate cu pretul anterior
    return s.rename(name)


def log_returns(name, start=None, end=END):
    """Randamente log pe calendarul propriu al seriei."""
    return np.log(load_close(name, start, end)).diff().dropna().rename(name)


rets = {k: log_returns(k) for k in MARKETS}
ORDER = list(MARKETS)
for k in ORDER:
    print(f'{LABELS[k]:26s}', rets[k].index[0].date(), '->', rets[k].index[-1].date(), len(rets[k]), 'obs')

## Efficiency tests

- `variance_ratio`: Lo–MacKinlay VR(q) with the homoskedastic $z$ and the robust $z^*$.
- `chow_denning`, `runs_test`, `robust_ljung_box`, `rs_hurst` (with Anis–Lloyd), `lo_modified_rs`, `dfa_hurst`, `rolling_stat`.

In [ ]:
def variance_ratio(r, q):
    """VR(q) cu suprapunere si corectiile de deplasare Lo-MacKinlay (1988).

    Returneaza (VR, z omoscedastic, z* robust la heteroscedasticitate).
    """
    x = np.asarray(r, dtype=float)
    T = len(x)
    mu = x.mean()
    e = x - mu
    var_a = (e ** 2).sum() / (T - 1)
    agg = np.convolve(x, np.ones(q), mode='valid') - q * mu        # sume pe q perioade, suprapuse
    m = q * (T - q + 1) * (1 - q / T)
    var_c = (agg ** 2).sum() / m
    vr = var_c / var_a
    z_homo = (vr - 1) / np.sqrt(2 * (2 * q - 1) * (q - 1) / (3 * q * T))
    den = (e ** 2).sum() ** 2
    theta = 0.0
    for j in range(1, q):
        delta = T * (e[j:] ** 2 * e[:-j] ** 2).sum() / den
        theta += (2 * (q - j) / q) ** 2 * delta
    z_star = np.sqrt(T) * (vr - 1) / np.sqrt(theta)
    return vr, z_homo, z_star


def chow_denning(r, qs=(2, 5, 10, 20)):
    """Testul multiplu Chow-Denning: max |z*(q)|; valoarea p din modulul maxim studentizat (df infinit)."""
    zs = [variance_ratio(r, q)[2] for q in qs]
    zmax = np.max(np.abs(zs))
    p = 1 - (2 * stats.norm.cdf(zmax) - 1) ** len(qs)
    return zmax, p, zs


def runs_test(r):
    """Numarul de secvente de semne (fara zerouri), media si varianta sub ipoteza de independenta."""
    s = np.sign(np.asarray(r, dtype=float))
    s = s[s != 0]
    n1, n2 = (s > 0).sum(), (s < 0).sum()
    n = n1 + n2
    runs = 1 + (s[1:] != s[:-1]).sum()
    mean = 2 * n1 * n2 / n + 1
    var = 2 * n1 * n2 * (2 * n1 * n2 - n) / (n ** 2 * (n - 1))
    z = (runs - mean) / np.sqrt(var)
    return runs, mean, z, 2 * (1 - stats.norm.cdf(abs(z)))


def robust_ljung_box(r, m=10):
    """Q(m) Ljung-Box si Q~(m) cu varianta robusta tau_k = sum e_t^2 e_{t-k}^2 / (sum e_t^2)^2."""
    e = np.asarray(r, dtype=float) - np.mean(r)
    T = len(e)
    s2 = (e ** 2).sum()
    q_lb, q_rob = 0.0, 0.0
    for k in range(1, m + 1):
        rho = (e[k:] * e[:-k]).sum() / s2
        tau = (e[k:] ** 2 * e[:-k] ** 2).sum() / s2 ** 2
        q_lb += T * (T + 2) * rho ** 2 / (T - k)
        q_rob += rho ** 2 / tau
    return q_lb, 1 - stats.chi2.cdf(q_lb, m), q_rob, 1 - stats.chi2.cdf(q_rob, m)


def _rs(x):
    y = np.cumsum(x - x.mean())
    s = x.std(ddof=0)
    return (y.max() - y.min()) / s if s > 0 else np.nan


def expected_rs(n):
    """E[R/S] pentru zgomot i.i.d. (Anis & Lloyd, 1976)."""
    if n <= 340:
        f = np.exp(gammaln((n - 1) / 2) - gammaln(n / 2)) / np.sqrt(np.pi)
    else:
        f = 1 / np.sqrt(n * np.pi / 2)
    i = np.arange(1, n)
    return f * np.sum(np.sqrt((n - i) / i))


def rs_hurst(r, min_n=10, n_sizes=20):
    """H din panta log(R/S) ~ log(n) (Hurst, 1951) si H corectat Anis-Lloyd (panta lui R/S - E[R/S] + 0,5)."""
    x = np.asarray(r, dtype=float)
    T = len(x)
    sizes = np.unique(np.logspace(np.log10(min_n), np.log10(T // 4), n_sizes).astype(int))
    rs, ers = [], []
    for n in sizes:
        k = T // n
        vals = [_rs(x[i * n:(i + 1) * n]) for i in range(k)]
        rs.append(np.nanmean(vals))
        ers.append(expected_rs(n))
    ls, lrs, lers = np.log(sizes), np.log(rs), np.log(ers)
    h = np.polyfit(ls, lrs, 1)[0]
    h_al = 0.5 + np.polyfit(ls, lrs - lers, 1)[0]
    return h, h_al, pd.DataFrame({'n': sizes, 'rs': rs, 'expected_rs': ers})


def lo_modified_rs(r, q=None):
    """Statistica V = Q_T / sqrt(T) a lui Lo (1991); sub H0 (fara memorie lunga) V in [0,809; 1,862] la 95%."""
    x = np.asarray(r, dtype=float)
    T = len(x)
    e = x - x.mean()
    if q is None:                                   # latimea de banda Andrews (1991) pentru AR(1)
        rho = np.corrcoef(e[1:], e[:-1])[0, 1]
        q = int(np.floor((3 * T / 2) ** (1 / 3) * (2 * rho / (1 - rho ** 2)) ** (2 / 3))) if rho > 0 else 0
    s2 = (e ** 2).mean()
    for j in range(1, q + 1):
        w = 1 - j / (q + 1)
        s2 += 2 * w * (e[j:] * e[:-j]).sum() / T
    y = np.cumsum(e)
    Q = (y.max() - y.min()) / np.sqrt(s2)
    return Q / np.sqrt(T), q


def dfa_hurst(r, min_box=10, n_sizes=18, max_frac=0.25):
    """Exponentul DFA-1: panta log F(n) ~ log n pe profilul cumulat al randamentelor centrate."""
    x = np.asarray(r, dtype=float)
    y = np.cumsum(x - x.mean())
    T = len(y)
    sizes = np.unique(np.logspace(np.log10(min_box), np.log10(int(T * max_frac)), n_sizes).astype(int))
    F = []
    for n in sizes:
        k = T // n
        seg = y[:k * n].reshape(k, n)
        t = np.arange(n)
        A = np.vstack([t, np.ones(n)]).T
        coef, *_ = np.linalg.lstsq(A, seg.T, rcond=None)
        resid = seg.T - A @ coef
        F.append(np.sqrt((resid ** 2).mean()))
    return np.polyfit(np.log(sizes), np.log(F), 1)[0], pd.DataFrame({'n': sizes, 'F': F})


def rolling_stat(r, func, window=500, step=21):
    """Aplica func pe ferestre mobile de `window` observatii, la fiecare `step` observatii."""
    x = r.values
    idx, out = [], []
    for end in range(window, len(x) + 1, step):
        idx.append(r.index[end - 1])
        out.append(func(x[end - window:end]))
    return pd.Series(out, index=idx)

## Seminar functions

- Calendar regressions with HAC errors and the functions used in Parts A–C (from `seminar2.py`).

In [ ]:
DAYS = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']
A_RETURNS = np.array([1.0, -0.5, 0.8, 0.3, -1.2, 0.6, 0.9, -0.4])   # %
A_SIGNS = '++-+++--+-++---+-+++'
A4_SIGNS = '+++++-----+++++-----'


def dow_regression(r):
    """Randament zilnic pe variabile dummy pentru zilele saptamanii (fara constanta), erori Newey-West."""
    X = pd.get_dummies(r.index.dayofweek).astype(float)
    X.columns = DAYS
    X.index = r.index
    res = sm.OLS(r * 1e4, X).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
    # testul egalitatii mediilor (Wald), cu aceleasi erori HAC
    R = np.zeros((4, 5))
    for i in range(4):
        R[i, 0], R[i, i + 1] = 1, -1
    w = res.wald_test(R, scalar=True)
    return res, float(w.pvalue)


def january_regression(r):
    m = np.exp(r.groupby(pd.Grouper(freq='ME')).sum()) - 1
    X = sm.add_constant(pd.Series((m.index.month == 1).astype(float), index=m.index, name='January'))
    res = sm.OLS(m * 100, X).fit(cov_type='HAC', cov_kwds={'maxlags': 3})
    return res


def turn_of_month(r):
    """Dummy pentru ultima zi de tranzactionare a lunii si primele trei zile ale lunii urmatoare (Ariel, 1987)."""
    d = pd.DataFrame({'r': r * 1e4})
    ym = d.index.to_period('M')
    rank = d.groupby(ym).cumcount()
    rank_end = d.groupby(ym).cumcount(ascending=False)
    d['tom'] = ((rank < 3) | (rank_end == 0)).astype(float)
    res = sm.OLS(d['r'], sm.add_constant(d['tom'])).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
    return res


def calendar_table():
    rows = []
    for k in ['sp500', 'bet', 'stoxx', 'wig20', 'bux']:
        r = rets[k]
        res, p_eq = dow_regression(r)
        jan = january_regression(r)
        tom = turn_of_month(r)
        rows.append(dict(market=LABELS[k], **{f'{d}_bp': res.params[d] for d in DAYS},
                         **{f'{d}_t': res.tvalues[d] for d in DAYS}, dow_equal_p=p_eq,
                         jan_extra_pct=jan.params['January'], jan_t=jan.tvalues['January'],
                         tom_extra_bp=tom.params['tom'], tom_t=tom.tvalues['tom'], tom_p=tom.pvalues['tom']))
    t = pd.DataFrame(rows).set_index('market')
    t.to_csv(os.path.join(TABLE_DIR, 'ch2_calendar_table.csv'), float_format='%.6g')
    return t


def a_vr_by_hand(r=A_RETURNS, q=2):
    """VR(q) fara corectii de deplasare: var(sume pe q zile, suprapuse) / (q * var zilnica)."""
    mu = r.mean()
    var1 = ((r - mu) ** 2).mean()
    sums = np.convolve(r, np.ones(q), mode='valid')
    varq = ((sums - q * mu) ** 2).mean()
    e = r - mu
    rho = [(e[j:] * e[:-j]).sum() / (e ** 2).sum() for j in range(1, q)]
    approx = 1 + 2 * sum((1 - j / q) * rho[j - 1] for j in range(1, q))
    return dict(mean=mu, var1=var1, sums=sums, varq=varq, VR=varq / (q * var1), rho=rho, approx=approx)


def a_runs(signs=A_SIGNS):
    n1, n2 = signs.count('+'), signs.count('-')
    n = n1 + n2
    runs = 1 + sum(signs[i] != signs[i - 1] for i in range(1, n))
    mean = 2 * n1 * n2 / n + 1
    var = 2 * n1 * n2 * (2 * n1 * n2 - n) / (n ** 2 * (n - 1))
    z = (runs - mean) / np.sqrt(var)
    return dict(n1=n1, n2=n2, runs=runs, mean=mean, var=var, z=z, p=2 * (1 - stats.norm.cdf(abs(z))))


def h_from_vr(vr, q):
    """Pentru cresteri autosimilare, VR(q) = q^(2H-1), deci H = 0,5 + ln VR(q) / (2 ln q)."""
    return 0.5 + np.log(vr) / (2 * np.log(q))


def b_vr_table(names):
    rows = []
    for k in names:
        r = log_returns(k) if k != 'bettr' else np.log(read_market('BETTR.INDX')['close']).diff().dropna()
        v2, v5, v10, v20 = (variance_ratio(r, q) for q in (2, 5, 10, 20))
        cd = chow_denning(r)
        rows.append(dict(market=LABELS.get(k, 'BET-TR (Romania)'), N=len(r),
                         VR2=v2[0], z2=v2[1], zstar2=v2[2], VR5=v5[0], z5=v5[1], zstar5=v5[2],
                         VR20=v20[0], zstar20=v20[2], CD=cd[0], CD_p=cd[1]))
    return pd.DataFrame(rows).set_index('market')


def b_rolling_dfa(k='sp500', window=500, step=21, n_null=300, seed=SEED):
    rng = np.random.default_rng(seed)
    null = [dfa_hurst(rng.standard_t(4, window))[0] for _ in range(n_null)]
    lo, hi = np.percentile(null, [2.5, 97.5])
    h = rolling_stat(log_returns(k), lambda x: dfa_hurst(x)[0], window, step)
    return dict(lo=lo, hi=hi, share_out=float(((h < lo) | (h > hi)).mean()), min=h.min(), max=h.max(),
                argmin=h.idxmin().date(), argmax=h.idxmax().date(), mean=h.mean())


def b_tom_ols_hac(k='sp500'):
    """Efectul turn-of-month: aceeasi regresie cu erori OLS clasice si cu erori HAC (Newey-West, 5 decalaje)."""
    r = log_returns(k) * 1e4
    d = pd.DataFrame({'r': r})
    ym = d.index.to_period('M')
    rank = d.groupby(ym).cumcount()
    rank_end = d.groupby(ym).cumcount(ascending=False)
    d['tom'] = ((rank < 3) | (rank_end == 0)).astype(float)
    X = sm.add_constant(d['tom'])
    ols = sm.OLS(d['r'], X).fit()
    hac = sm.OLS(d['r'], X).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
    return dict(N=len(d), share_tom=float(d['tom'].mean()), const=float(ols.params['const']),
                beta=float(ols.params['tom']), se_ols=float(ols.bse['tom']), t_ols=float(ols.tvalues['tom']),
                se_hac=float(hac.bse['tom']), t_hac=float(hac.tvalues['tom']), p_hac=float(hac.pvalues['tom']),
                mean_tom=float(d.loc[d['tom'] == 1, 'r'].mean()), mean_other=float(d.loc[d['tom'] == 0, 'r'].mean()))

## Part A — pen and paper

### A1. Variance ratio, step by step [Solved]

- Returns (%): 1.0, −0.5, 0.8, 0.3, −1.2, 0.6, 0.9, −0.4.
- Compute $\hat\mu$, $\hat\sigma^2_1$, the overlapping 2-day sums, $\hat\sigma^2_2$ and $\widehat{\text{VR}}(2)$; compare with $1 + \hat\rho_1$.

In [6]:
# Solution
a1 = a_vr_by_hand()
{k: np.round(v, 4) for k, v in a1.items()}

{'mean': np.float64(0.1875),
 'var1': np.float64(0.5586),
 'sums': array([ 0.5,  0.3,  1.1, -0.9, -0.6,  1.5,  0.5]),
 'varq': np.float64(0.6292),
 'VR': np.float64(0.5632),
 'rho': array([-0.3947]),
 'approx': np.float64(0.6053)}

### A2. Variance ratio at three days [Proposed]

- Same returns, $q = 3$: sums, $\widehat{\text{VR}}(3)$, $\hat\rho_2$ and $1 + 2(\frac23\hat\rho_1 + \frac13\hat\rho_2)$. Model: A1.

In [ ]:
# Your code here


### A3. Runs test, step by step [Solved]

- Signs: `++-+++--+-++---+-+++`; count $n_1$, $n_2$, $R$; compute $z$ and the $p$-value.

In [8]:
# Solution
a_runs()

{'n1': 12,
 'n2': 8,
 'runs': 11,
 'mean': 10.6,
 'var': 4.3452631578947365,
 'z': np.float64(0.1918898262711097),
 'p': np.float64(0.8478285068142399)}

### A4. A sequence with momentum [Proposed]

- Signs: `+++++-----+++++-----`. Model: A3.

In [ ]:
# Your code here


### A5. Hurst exponent from a variance ratio [Solved]

- $H = 0.5 + \ln\text{VR}(q)/(2\ln q)$; BET $\widehat{\text{VR}}(20)$.

In [10]:
# Solution
vr_bet = variance_ratio(rets['bet'], 20)[0]
print(round(vr_bet, 4), round(h_from_vr(vr_bet, 20), 4))

1.377 0.5534


### A6. Hurst exponent of the S&P 500 [Proposed]

- Same for the S&P 500; then the VR(20) of an AR(1) with $\rho_1 = -0.1$. Model: A5.

In [ ]:
# Your code here


## Part B — computation and inference

### B1. Homoskedastic vs robust variance ratio [Solved]

- S&P 500 and BET: VR(q), $z$ and $z^*$ for $q = 2, 5, 20$; Chow–Denning.
- Interpretation: why is $|z| > |z^*|$? Would you trade on the S&P 500 reversal?

In [12]:
# Solution
b_vr_table(['sp500', 'bet']).round(3).T

market,S&P 500 (US),BET (Romania)
N,6714.000,6670.000
VR2,0.901,1.108
z2,-8.120,8.827
zstar2,-3.644,3.836
VR5,0.830,1.173
z5,-6.375,6.445
zstar5,-2.772,2.980
VR20,0.753,1.377
zstar20,-1.856,3.311
CD,3.644,3.836


### B2. BET-TR and emerging markets [Proposed]

- Repeat B1 for BET-TR, WIG20, BUX, PX, BIST 100, Bovespa, IPC, Nifty 50, Shanghai. Model: B1.
- Interpretation: how many reject with $z(2)$, how many with Chow–Denning?

In [ ]:
# Your code here


### B3. Rolling DFA on the S&P 500 [Solved]

- 500-day windows, step 21; null band from 300 i.i.d. Student-$t_4$ series.
- Interpretation: is the share outside the band above 5%?

In [14]:
# Solution
b_rolling_dfa()

{'lo': np.float64(0.3778242224128016),
 'hi': np.float64(0.6143493884737028),
 'share_out': 0.06418918918918919,
 'min': 0.34072667671889256,
 'max': 0.5978900709448012,
 'argmin': datetime.date(2022, 3, 15),
 'argmax': datetime.date(2021, 2, 12),
 'mean': np.float64(0.4501423601066471)}

### B4. Adaptive efficiency of Bitcoin [Proposed]

- Bitcoin from 2011, 365-day windows, step 30; $z^*(5)$ with a wild-bootstrap band per window. Model: B3, B1.
- Interpretation: how many rejections would chance produce?

In [ ]:
# Your code here


### B5. Turn of the month with HAC errors [Solved]

- $r_t = a + b\,\text{TOM}_t + u_t$ with OLS and HAC (5 lags) errors, S&P 500 and BET.
- Interpretation: why do OLS and HAC errors differ?

In [16]:
# Solution
pd.DataFrame({k: b_tom_ols_hac(k) for k in ['sp500', 'bet']}).round(4)

,sp500,bet
N,6714.0000,6670.0000
share_tom,0.1912,0.1925
const,1.8356,2.9627
beta,3.3271,17.8201
se_ols,3.7643,4.3385
t_ols,0.8839,4.1075
se_hac,3.4850,4.7693
t_hac,0.9547,3.7364
p_hac,0.3397,0.0002
mean_tom,5.1627,20.7828


### B6. Fifteen calendar tests and multiple testing [Proposed]

- Five markets × three effects; raw, Bonferroni and Holm decisions. Model: B5.

In [ ]:
# Your code here


### B7. Is the S&P 500 reversal a crisis effect? [Proposed]

- $\hat\rho_1$, i.i.d. and robust $t$, $z^*(5)$, with and without Sep 2008–Jun 2009 and 15 Feb–30 Jun 2020. Model: B1.

In [ ]:
# Your code here


## Part C — did the BVB become more efficient after the 2020 FTSE upgrade? [Proposed]

- Five years before and after 21 September 2020: $\hat\rho_1$, $z^*(5)$ and a block-bootstrap interval for the change.

In [ ]:
# Your code here
